Setup env

In [ ]:
pip install nltk gensim scikit-learn numpy seaborn pyspellchecker gensim

In [ ]:
pip install --user --upgrade scipy

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng') 
# nltk.download('resource_name')

Data Exploration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from spellchecker import SpellChecker
from nltk import pos_tag
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import CountVectorizer
import gensim



In [ ]:
def load_tweet_file(path):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    if '\n' in content.strip() and content.count('\n') > 5:
        tweets = [line.strip() for line in content.split('\n') if line.strip()]
    else:
        tweets = [t.strip() for t in content.split(',') if t.strip()]
    return pd.DataFrame({'tweets': tweets})

neg = load_tweet_file('../data/processedNegative.csv')
pos = load_tweet_file('../data/processedPositive.csv')
neu = load_tweet_file('../data/processedNeutral.csv')

print(neg.shape, pos.shape, neu.shape)


In [ ]:
neg['label'] = 'Negative'
pos['label'] = 'Positive'
neu['label'] = 'Neutral'

df = pd.concat([neg, pos, neu], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
print(df['label'].value_counts())

df.head()

df['label'].value_counts().plot(kind='bar')

In [ ]:
df['char_length'] = df['tweets'].apply(len)

df['word_count'] = df['tweets'].apply(lambda x: len(x.split()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['char_length'], bins=50, ax=axes[0])
axes[0].set_title('Distribution longueur (caractères)')
axes[0].set_xlabel('Nombre de caractères')

sns.histplot(df['word_count'], bins=50, ax=axes[1])
axes[1].set_title('Distribution longueur (mots)')
axes[1].set_xlabel('Nombre de mots')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='label', y='word_count')
plt.title('Longueur des tweets par label')
plt.show()

In [ ]:
n_dup = df.duplicated(subset='tweets').sum()

duplicated_tweets = df[df.duplicated(subset='tweets', keep=False)].sort_values('tweets')
duplicated_tweets.head(10)

duplicates_check = df[df.duplicated(subset='tweets', keep=False)]
label_consistency = duplicates_check.groupby('tweets')['label'].nunique()
inconsistent = label_consistency[label_consistency > 1]

In [ ]:
def basic_clean(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
df_clean = df.copy()
df_clean['tweets'] = df_clean['tweets'].fillna('')
df_clean = df_clean.dropna(subset=['tweets'])
df_clean['clean_tweets'] = df_clean['tweets'].apply(basic_clean)

df_clean = df_clean.drop_duplicates(subset='clean_tweets', keep='first').reset_index(drop=True)

print(df_clean.shape)

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [ ]:
def tokenize_only(text):
    tokens = word_tokenize(text)
    return ' '.join(tokens)

df_clean['tokenized'] = df_clean['clean_tweets'].apply(tokenize_only)

In [ ]:
def apply_stemming(text):
    tokens = word_tokenize(text)
    stemmed = [stemmer.stem(t) for t in tokens]
    return ' '.join(stemmed)

df_clean['stemmed'] = df_clean['clean_tweets'].apply(apply_stemming)

In [ ]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def apply_lemmatization(text):
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    lemmatized = [lemmatizer.lemmatize(t, get_wordnet_pos(tag)) for t, tag in tagged]
    return ' '.join(lemmatized)

df_clean['lemmatized'] = df_clean['clean_tweets'].apply(apply_lemmatization)

In [ ]:
spell = SpellChecker()

def correct_spelling(text):
    tokens = text.split()
    corrected = []
    for t in tokens:
        c = spell.correction(t)
        corrected.append(c if c is not None else t)
    return ' '.join(corrected)

df_clean['spell_corrected'] = df_clean['clean_tweets'].apply(correct_spelling)

In [ ]:
df_clean['stemmed_misspellings'] = df_clean['spell_corrected'].apply(apply_stemming)
df_clean['lemmatized_misspellings'] = df_clean['spell_corrected'].apply(apply_lemmatization)

In [ ]:
def remove_stopwords(text):
    tokens = text.split()
    filtered = [t for t in tokens if t not in stop_words]
    return ' '.join(filtered)

df_clean['lemmatized_no_stopwords'] = df_clean['lemmatized'].apply(remove_stopwords)

In [ ]:
preprocessing_variants = {
    'tokenization': df_clean['tokenized'],
    'stemming': df_clean['stemmed'],
    'lemmatization': df_clean['lemmatized'],
    'stemming_misspellings': df_clean['stemmed_misspellings'],
    'lemmatization_misspellings': df_clean['lemmatized_misspellings'],
    'lemmatization_no_stopwords': df_clean['lemmatized_no_stopwords'],
}

Similarity

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorization_types = ['binary', 'count', 'tfidf']
matrix_table = pd.DataFrame(index=preprocessing_variants.keys(), columns=vectorization_types)

for name, series in preprocessing_variants.items():
    ngram_range = (1, 2) if name == 'ngrams_bigram' else (1, 1)

    binary_vec = CountVectorizer(binary=True, ngram_range=ngram_range)
    X_binary = binary_vec.fit_transform(series)

    count_vec = CountVectorizer(binary=False, ngram_range=ngram_range)
    X_count = count_vec.fit_transform(series)

    tfidf_vec = TfidfVectorizer(ngram_range=ngram_range)
    X_tfidf = tfidf_vec.fit_transform(series)

    matrix_table.loc[name, 'binary'] = X_binary
    matrix_table.loc[name, 'count'] = X_count
    matrix_table.loc[name, 'tfidf'] = X_tfidf

In [ ]:
tfidf_matrix = matrix_table.loc['lemmatization_no_stopwords']['tfidf']
tfidf_similarity = cosine_similarity(tfidf_matrix)

In [ ]:
def get_top_similar_pairs(similarity_matrix, texts, top_n=10):
    upper_triangle = np.triu(similarity_matrix, k=1)
    flat_indices = np.argsort(upper_triangle, axis=None)[::-1][:top_n]
    row_indices, col_indices = np.unravel_index(flat_indices, upper_triangle.shape)

    results = []
    for i, j in zip(row_indices, col_indices):
        results.append({
            'tweet_1': texts.iloc[i],
            'tweet_2': texts.iloc[j],
            'similarity': upper_triangle[i, j]
        })
    return pd.DataFrame(results)

all_top_pairs = {}

for name, texts_series in preprocessing_variants.items():
    tfidf = matrix_table.loc[name, 'tfidf']
    tfidf_similarity = cosine_similarity(tfidf)

    top10_df = get_top_similar_pairs(tfidf_similarity, texts_series, top_n=10)
    all_top_pairs[name] = top10_df

    print(f"--- Top 10 Similar Pairs for: {name} ---")
    display(top10_df)

In [ ]:
mask = df_clean['lemmatized_no_stopwords'] == 'miss unhappy'
print(df_clean.loc[mask, 'tweets'].tolist())

In [ ]:
heatmap_data = []
for name, top10_df in all_top_pairs.items():
    temp_df = top10_df.copy()
    temp_df['variant'] = name
    temp_df['pair_rank'] = range(1, len(temp_df) + 1)
    heatmap_data.append(temp_df[['variant', 'pair_rank', 'similarity']])

combined_df = pd.concat(heatmap_data)

pivot_df = combined_df.pivot(index='variant', columns='pair_rank', values='similarity')

plt.figure(figsize=(12, 6))
sns.heatmap(
    pivot_df, 
    annot=True,          
    fmt=".2f",           
    cmap="YlGnBu",
    cbar_kws={'label': 'Cosine Similarity Score'},
    linewidths=.5,
    vmin=0, vmax=1.0
)

plt.title('Heatmap of Top Tweet Pair Similarities Across Variants', fontsize=14, fontweight='bold')
plt.xlabel('Tweet Pair Rank (1 to 10)', fontsize=11)
plt.ylabel('Variant Name', fontsize=11)
plt.tight_layout()
plt.show()

Train ~ Test Split

In [48]:
from sklearn.model_selection import  train_test_split

X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, df_clean['label'], test_size=0.2, random_state=42)


Machine learning

In [49]:
from sklearn.linear_model import  LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import  accuracy_score

rfc = RandomForestClassifier()
lgr = LogisticRegression()

models = [rfc, lgr]

for model in models:
        
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test,y_pred)
    print(model)
    print(f'Accuracy Score: {acc}')
    print()

RandomForestClassifier()
Accuracy Score: 0.8607038123167156

LogisticRegression()
Accuracy Score: 0.8504398826979472



In [ ]:
from gensim import corpora
from gensim.utils import simple_preprocess

texts = df_clean['tokenized']
tokenized = [simple_preprocess(doc) for doc in texts]

dictionary = corpora.Dictionary(tokenized)

corpus = [dictionary.doc2bow(text) for text in tokenized]


In [ ]:
import gensim.downloader as api
glove_model = api.load("glove-twitter-100")

[==================================================] 100.0% 387.1/387.1MB downloaded


In [43]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=tokenized,
    vector_size=300,   
    window=5,          
    min_count=1,       
    sg=1,            
    workers=4,
    epochs=10
)

model.save("word2vec.model")

In [54]:
def document_vector(tokens, model):
    valid_words = [w for w in tokens if w in model.key_to_index]
    if len(valid_words) == 0:
        return np.zeros(model.vector_size)
    return np.mean(model[valid_words], axis=0)

X_w2v = np.array([document_vector(tokens, glove_model) for tokens in tokenized])

X_train, X_test, y_train, y_test = train_test_split(X_w2v, df_clean['label'], test_size=0.2, random_state=42)

In [55]:
from sklearn.linear_model import  LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import  accuracy_score

rfc = RandomForestClassifier()
lgr = LogisticRegression()

models = [rfc, lgr]

for model in models:
        
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test,y_pred)
    print(model)
    print(f'Accuracy Score: {acc}')
    print()

RandomForestClassifier()
Accuracy Score: 0.8313782991202346

LogisticRegression()
Accuracy Score: 0.8651026392961877

